# Проверка Excel перед загрузкой в PostgreSQL

Читает `.xlsx`, где **имена листов = таблицы БД**, **столбцы = поля таблицы**.

Проверяет: листы/столбцы, NOT NULL, UUID, **FK внутри файла** (ошибка `23503`), дубликаты id/username/email.

**Порядок вставки:** `clients` → `employees` → `tickets` → `attachments` → `reviews` → `notifications` → `chat_history`

```bash
pip install pandas openpyxl
```

In [17]:
from __future__ import annotations

import re
import uuid
from pathlib import Path
from typing import Any

import pandas as pd

# === НАСТРОЙКИ: путь к вашему Excel ===
EXCEL_PATH = Path("nerd_analytics_v25.xlsx")  # измените на свой файл

# Листы в порядке загрузки (как в БД)
TABLE_ORDER = [
    "clients",
    "employees",
    "tickets",
    "attachments",
    "reviews",
    "notifications",
    "chat_history",
]

# Схема: столбцы и nullable (False = обязательно)
SCHEMA: dict[str, dict[str, bool]] = {
    "clients": {
        "id": False,
        "username": False,
        "email": False,
        "full_name": True,
        "age": True,
        "gender": True,
        "city": True,
        "password_hash": False,
        "created_at": True,
    },
    "employees": {
        "id": False,
        "username": False,
        "full_name": False,
        "birthday": True,
        "phone": True,
        "sec_level": True,
        "status": True,
        "password_hash": False,
        "created_at": True,
    },
    "tickets": {
        "id": False,
        "client_id": False,
        "responsible_id": True,
        "product": False,
        "status": True,
        "priority": True,
        "date": False,
        "deadline": False,
        "closed_at": True,
        "reopened_count": True,
        "last_reopened_at": True,
        "ai_suggested_category": True,
        "final_category": True,
        "is_admin_changed": True,
        "keywords": True,
        "confidence": True,
        "sla_ttfr_min": True,
        "sla_ttr_min": True,
    },
    "attachments": {
        "id": False,
        "ticket_id": False,
        "file_url": False,
        "file_type": True,
        "created_at": True,
    },
    "reviews": {
        "id": False,
        "ticket_id": True,
        "client_id": False,
        "product": True,
        "rating": False,
        "comment": True,
        "ai_suggested_category": True,
        "final_category": True,
        "is_admin_changed": True,
        "sentiment": True,
        "keywords_positive": True,
        "keywords_neutral": True,
        "keywords_negative": True,
        "confidence": True,
        "created_at": True,
    },
    "notifications": {
        "id": False,
        "client_id": False,
        "ticket_id": False,
        "type": False,
        "status": True,
        "created_at": True,
    },
    "chat_history": {
        "id": False,
        "chat_id": False,
        "ticket_id": True,
        "client_id": False,
        "role": False,
        "product": True,
        "category": True,
        "resolved_by_ai": True,
        "message": False,
        "created_at": True,
    },
}

# FK: column -> parent table.column
FOREIGN_KEYS: dict[str, list[tuple[str, str, str]]] = {
    "tickets": [
        ("client_id", "clients", "id"),
        ("responsible_id", "employees", "id"),
    ],
    "attachments": [("ticket_id", "tickets", "id")],
    "reviews": [
        ("ticket_id", "tickets", "id"),
        ("client_id", "clients", "id"),
    ],
    "notifications": [
        ("client_id", "clients", "id"),
        ("ticket_id", "tickets", "id"),
    ],
    "chat_history": [
        ("ticket_id", "tickets", "id"),
        ("client_id", "clients", "id"),
    ],
}

UUID_RE = re.compile(
    r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$",
    re.I,
)

In [16]:
def _is_empty(val: Any) -> bool:
    if val is None:
        return True
    if isinstance(val, float) and pd.isna(val):
        return True
    if isinstance(val, str) and not val.strip():
        return True
    return False


def _norm_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def _parse_uuid(val: Any) -> str | None:
    if _is_empty(val):
        return None
    s = str(val).strip()
    if UUID_RE.match(s):
        return s.lower()
    try:
        return str(uuid.UUID(s)).lower()
    except (ValueError, AttributeError):
        return None


class ValidationReport:
    def __init__(self) -> None:
        self.errors: list[str] = []
        self.warnings: list[str] = []

    def error(self, msg: str) -> None:
        self.errors.append(msg)

    def warn(self, msg: str) -> None:
        self.warnings.append(msg)

    @property
    def ok(self) -> bool:
        return len(self.errors) == 0

    def summary(self) -> str:
        lines = [
            f"Ошибок: {len(self.errors)}",
            f"Предупреждений: {len(self.warnings)}",
            "Статус: OK" if self.ok else "Статус: ЕСТЬ ОШИБКИ — в БД заливать нельзя",
        ]
        if self.errors:
            lines.append("\n--- Ошибки ---")
            lines.extend(f"  • {e}" for e in self.errors)
        if self.warnings:
            lines.append("\n--- Предупреждения ---")
            lines.extend(f"  • {w}" for w in self.warnings)
        return "\n".join(lines)


def validate_table(
    table: str,
    df: pd.DataFrame,
    report: ValidationReport,
    loaded: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    if table not in SCHEMA:
        report.error(f"Неизвестная таблица: {table}")
        return df

    expected = SCHEMA[table]
    df = _norm_columns(df)

    # лишние / недостающие столбцы
    exp_cols = set(expected.keys())
    got_cols = set(df.columns)
    missing = exp_cols - got_cols
    extra = got_cols - exp_cols
    if missing:
        report.error(f"[{table}] нет столбцов: {sorted(missing)}")
    if extra:
        report.warn(f"[{table}] лишние столбцы (будут проигнорированы): {sorted(extra)}")
    df = df[[c for c in expected if c in df.columns]]

    if df.empty:
        report.warn(f"[{table}] лист пустой")
        return df

    # NOT NULL
    for col, nullable in expected.items():
        if col not in df.columns:
            continue
        if nullable:
            continue
        bad = df[col].apply(_is_empty)
        if bad.any():
            rows = df.index[bad].tolist()[:10]
            report.error(
                f"[{table}] пустые обязательные значения в '{col}', строки (0-based): {rows}"
            )

    # UUID в id и *_id (+ NaN → DBeaver пишет "nan" в uuid = ошибка 22P02)
    for col in df.columns:
        if col == "id" or col.endswith("_id"):
            nan_rows = []
            for idx, val in df[col].items():
                if isinstance(val, float) and pd.isna(val):
                    if not expected.get(col, True):
                        report.error(f"[{table}] строка {idx}: пустой обязательный {col}")
                    else:
                        nan_rows.append(idx)
                    continue
                if _is_empty(val):
                    if not expected.get(col, True):
                        report.error(f"[{table}] строка {idx}: пустой {col}")
                    continue
                s = str(val).strip().lower()
                if s in ("nan", "none", "null", "#n/a"):
                    report.error(
                        f"[{table}] строка {idx}: {col}={val!r} — для UUID нужен NULL, не текст 'nan'"
                    )
                    continue
                parsed = _parse_uuid(val)
                if parsed is None:
                    report.error(f"[{table}] строка {idx}: невалидный UUID в {col}: {val!r}")
            if nan_rows:
                report.warn(
                    f"[{table}] {col}: {len(nan_rows)} ячеек NaN (пусто) — при импорте в DBeaver "
                    f"должен быть NULL; иначе SQL Error 22P02. Первая строка (0-based): {nan_rows[0]}"
                )

    # Типы для reviews (и похожих)
    if table == "reviews":
        for idx, val in df.get("rating", pd.Series(dtype=object)).items():
            if _is_empty(val):
                report.error(f"[reviews] строка {idx}: пустой rating")
            else:
                try:
                    r = int(float(val))
                    if r < 1 or r > 5:
                        report.error(f"[reviews] строка {idx}: rating={val!r} вне 1–5")
                except (TypeError, ValueError):
                    report.error(f"[reviews] строка {idx}: rating не число: {val!r}")
        for idx, val in df.get("confidence", pd.Series(dtype=object)).items():
            if _is_empty(val):
                continue
            try:
                float(str(val).replace(",", "."))
            except (TypeError, ValueError):
                report.error(f"[reviews] строка {idx}: confidence не число: {val!r}")
        for idx, val in df.get("is_admin_changed", pd.Series(dtype=object)).items():
            if _is_empty(val):
                continue
            if isinstance(val, bool):
                continue
            if str(val).strip().lower() not in {
                "true", "false", "1", "0", "yes", "no", "t", "f",
            }:
                report.error(f"[reviews] строка {idx}: is_admin_changed не bool: {val!r}")

    # дубликаты id
    if "id" in df.columns:
        ids = df["id"].dropna().astype(str)
        dup = ids[ids.duplicated()]
        if not dup.empty:
            report.error(f"[{table}] дубликаты id: {dup.unique()[:5].tolist()}")

    # уникальность clients
    if table == "clients":
        for col in ("username", "email"):
            if col in df.columns:
                dup = df[col].dropna()
                dup = dup[dup.duplicated()]
                if not dup.empty:
                    report.error(f"[{table}] дубликаты {col}: {dup.unique()[:5].tolist()}")

    # FK внутри Excel (накопленные листы + текущий)
    if table in FOREIGN_KEYS:
        for fk_col, parent_table, parent_col in FOREIGN_KEYS[table]:
            if fk_col not in df.columns:
                continue
            parent_df = loaded.get(parent_table)
            if parent_df is None:
                report.error(
                    f"[{table}] нет листа '{parent_table}' для проверки FK {fk_col}"
                )
                continue
            parent_ids = set(
                parent_df[parent_col].dropna().apply(lambda x: _parse_uuid(x)).dropna()
            )
            for idx, val in df[fk_col].items():
                if _is_empty(val):
                    if expected.get(fk_col, True):
                        continue
                    continue
                uid = _parse_uuid(val)
                if uid is None:
                    continue
                if uid not in parent_ids:
                    report.error(
                        f"[{table}] строка {idx}: {fk_col}={uid} "
                        f"нет в {parent_table}.{parent_col} (ошибка 23503 в БД)"
                    )

    return df


def load_and_validate(path: Path) -> tuple[dict[str, pd.DataFrame], ValidationReport]:
    report = ValidationReport()
    if not path.exists():
        report.error(f"Файл не найден: {path.resolve()}")
        return {}, report

    xl = pd.ExcelFile(path)
    sheet_map = {s.strip().lower(): s for s in xl.sheet_names}

    for table in TABLE_ORDER:
        if table not in sheet_map:
            report.warn(f"Лист '{table}' отсутствует в Excel (пропуск)")

    loaded: dict[str, pd.DataFrame] = {}
    for table in TABLE_ORDER:
        if table not in sheet_map:
            continue
        raw = pd.read_excel(path, sheet_name=sheet_map[table], dtype=object)
        loaded[table] = validate_table(table, raw, report, loaded)

    # лишние листы
    for s in sheet_map:
        if s not in TABLE_ORDER:
            report.warn(f"Лишний лист в Excel: '{sheet_map[s]}'")

    return loaded, report


data, report = load_and_validate(EXCEL_PATH)
print(report.summary())

# краткая сводка по листам
for name in TABLE_ORDER:
    if name in data:
        print(f"{name}: {len(data[name])} строк, столбцы: {list(data[name].columns)}")
# === reviews: подготовка ticket_id NaN → None (для импорта без 22P02) ===
if "reviews" in data:
    rv = data["reviews"].copy()
    if "ticket_id" in rv.columns:
        before = rv["ticket_id"].isna().sum()
        rv["ticket_id"] = rv["ticket_id"].where(pd.notna(rv["ticket_id"]), None)
        print(f"reviews: ticket_id пустых (→ NULL): {before}")
    # рейтинг целым числом
    if "rating" in rv.columns:
        rv["rating"] = rv["rating"].apply(lambda x: int(float(x)) if not _is_empty(x) else x)
    out = Path("data/reviews_for_import.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    rv.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"Сохранено для DBeaver: {out.resolve()}")
    print("Импортируйте CSV: пустой ticket_id = пустая ячейка → NULL")

# Детально: chat_history (частая причина 23503)
if "chat_history" in data:
    ch = data["chat_history"]


    def _id_set(table: str, col: str = "id") -> set[str]:
        if table not in data or col not in data[table].columns:
            return set()
        return {u for u in (_parse_uuid(v) for v in data[table][col]) if u}


    client_ids = _id_set("clients")
    ticket_ids = _id_set("tickets")

    print("client_id в clients:", len(client_ids), "| ticket_id в tickets:", len(ticket_ids))
    for idx, row in ch.iterrows():
        cid = _parse_uuid(row.get("client_id"))
        if not cid:
            print(f"строка {idx}: пустой/битый client_id")
        elif cid not in client_ids:
            print(f"строка {idx}: client_id {cid} → нет в листе clients")
        tid = _parse_uuid(row.get("ticket_id"))
        if tid and tid not in ticket_ids:
            print(f"строка {idx}: ticket_id {tid} → нет в листе tickets")

Ошибок: 0
Предупреждений: 2
Статус: OK

--- Предупреждения ---
  • [reviews] ticket_id: 500 ячеек NaN (пусто) — при импорте в DBeaver должен быть NULL; иначе SQL Error 22P02. Первая строка (0-based): 735
  • [chat_history] ticket_id: 3911 ячеек NaN (пусто) — при импорте в DBeaver должен быть NULL; иначе SQL Error 22P02. Первая строка (0-based): 1
clients: 500 строк, столбцы: ['id', 'username', 'email', 'full_name', 'age', 'gender', 'city', 'password_hash', 'created_at']
employees: 30 строк, столбцы: ['id', 'username', 'full_name', 'birthday', 'phone', 'sec_level', 'status', 'password_hash', 'created_at']
tickets: 3000 строк, столбцы: ['id', 'client_id', 'responsible_id', 'product', 'status', 'priority', 'date', 'deadline', 'closed_at', 'reopened_count', 'last_reopened_at', 'ai_suggested_category', 'final_category', 'is_admin_changed', 'keywords', 'confidence', 'sla_ttfr_min', 'sla_ttr_min']
attachments: 1000 строк, столбцы: ['id', 'ticket_id', 'file_url', 'file_type', 'created_at']
rev

In [11]:
data, report = load_and_validate(EXCEL_PATH)
print(report.summary())

# краткая сводка по листам
for name in TABLE_ORDER:
    if name in data:
        print(f"{name}: {len(data[name])} строк, столбцы: {list(data[name].columns)}")

Ошибок: 0
Предупреждений: 2
Статус: OK

--- Предупреждения ---
  • [reviews] ticket_id: 500 ячеек NaN (пусто) — при импорте в DBeaver должен быть NULL; иначе SQL Error 22P02. Первая строка (0-based): 735
  • [chat_history] ticket_id: 3911 ячеек NaN (пусто) — при импорте в DBeaver должен быть NULL; иначе SQL Error 22P02. Первая строка (0-based): 1
clients: 500 строк, столбцы: ['id', 'username', 'email', 'full_name', 'age', 'gender', 'city', 'password_hash', 'created_at']
employees: 30 строк, столбцы: ['id', 'username', 'full_name', 'birthday', 'phone', 'sec_level', 'status', 'password_hash', 'created_at']
tickets: 3000 строк, столбцы: ['id', 'client_id', 'responsible_id', 'product', 'status', 'priority', 'date', 'deadline', 'closed_at', 'reopened_count', 'last_reopened_at', 'ai_suggested_category', 'final_category', 'is_admin_changed', 'keywords', 'confidence', 'sla_ttfr_min', 'sla_ttr_min']
attachments: 1000 строк, столбцы: ['id', 'ticket_id', 'file_url', 'file_type', 'created_at']
rev

In [12]:
# === reviews: подготовка ticket_id NaN → None (для импорта без 22P02) ===
if "reviews" in data:
    rv = data["reviews"].copy()
    if "ticket_id" in rv.columns:
        before = rv["ticket_id"].isna().sum()
        rv["ticket_id"] = rv["ticket_id"].where(pd.notna(rv["ticket_id"]), None)
        print(f"reviews: ticket_id пустых (→ NULL): {before}")
    # рейтинг целым числом
    if "rating" in rv.columns:
        rv["rating"] = rv["rating"].apply(lambda x: int(float(x)) if not _is_empty(x) else x)
    out = Path("data/reviews_for_import.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    rv.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"Сохранено для DBeaver: {out.resolve()}")
    print("Импортируйте CSV: пустой ticket_id = пустая ячейка → NULL")

# Детально: chat_history (частая причина 23503)
if "chat_history" in data:
    ch = data["chat_history"]

    def _id_set(table: str, col: str = "id") -> set[str]:
        if table not in data or col not in data[table].columns:
            return set()
        return {u for u in (_parse_uuid(v) for v in data[table][col]) if u}

    client_ids = _id_set("clients")
    ticket_ids = _id_set("tickets")

    print("client_id в clients:", len(client_ids), "| ticket_id в tickets:", len(ticket_ids))
    for idx, row in ch.iterrows():
        cid = _parse_uuid(row.get("client_id"))
        if not cid:
            print(f"строка {idx}: пустой/битый client_id")
        elif cid not in client_ids:
            print(f"строка {idx}: client_id {cid} → нет в листе clients")
        tid = _parse_uuid(row.get("ticket_id"))
        if tid and tid not in ticket_ids:
            print(f"строка {idx}: ticket_id {tid} → нет в листе tickets")

reviews: ticket_id пустых (→ NULL): 500
Сохранено для DBeaver: C:\Users\Ольчик\PycharmProjects\nerd-analytics-platform\backend\notebooks\data\reviews_for_import.csv
Импортируйте CSV: пустой ticket_id = пустая ячейка → NULL
client_id в clients: 500 | ticket_id в tickets: 3000


## Опционально: сверка с уже залитой БД

Раскомментируйте ячейку ниже, если часть данных уже в Postgres (`backend/.env` → `DATABASE_URL`).

In [13]:
# import asyncio
# from sqlalchemy import text
# from sqlalchemy.ext.asyncio import create_async_engine
# import sys
# sys.path.insert(0, str(Path.cwd().parent.parent))  # корень репо
# from backend.app.config import get_settings
#
# async def check_db_fk():
#     engine = create_async_engine(get_settings().DATABASE_URL)
#     async with engine.connect() as conn:
#         if "chat_history" not in data:
#             return
#         for idx, row in data["chat_history"].iterrows():
#             cid = _parse_uuid(row.get("client_id"))
#             if cid:
#                 r = await conn.execute(text("SELECT 1 FROM clients WHERE id = :id"), {"id": cid})
#                 if r.first() is None:
#                     print(f"DB: строка {idx} client_id {cid} нет в clients")
#     await engine.dispose()
#
# # asyncio.run(check_db_fk())

## Загрузка в PostgreSQL (рекомендуется вместо DBeaver)

Корректно ставит **NULL** для пустого `ticket_id` (без `22P02`) и все обязательные поля (без `23502`).

**DBeaver:** в маппинге импорта обязательно включите колонку **`is_admin_changed`** (или примените `alembic upgrade head` — миграция 005 даёт DEFAULT `false`).

1. Postgres + `alembic upgrade head` (миграция **005**)
2. Validate — **Статус: OK**
3. `TABLES_TO_LOAD` → ячейка ниже

`pip install psycopg2-binary`

In [ ]:
import sys

from sqlalchemy import create_engine, text

# корень репозитория (…/nerd-analytics-platform)
REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from backend.app.config import get_settings

# Что залить: только reviews или весь порядок
TABLES_TO_LOAD = ["reviews"]  # или TABLE_ORDER для всех таблиц
TRUNCATE_BEFORE = True  # очистить таблицу перед вставкой


def _cell_uuid(val: Any) -> str | None:
    if isinstance(val, float) and pd.isna(val):
        return None
    if _is_empty(val):
        return None
    return _parse_uuid(val)


def prepare_for_db(df: pd.DataFrame, table: str) -> pd.DataFrame:
    cols = [c for c in SCHEMA[table] if c in df.columns]
    out_rows: list[dict] = []
    for _, row in df.iterrows():
        rec: dict[str, Any] = {}
        for col in cols:
            val = row[col]
            if col in ("id",) or col.endswith("_id"):
                rec[col] = _cell_uuid(val)
            elif col in ("is_admin_changed", "resolved_by_ai"):
                if _is_empty(val):
                    rec[col] = False
                elif isinstance(val, bool):
                    rec[col] = val
                else:
                    rec[col] = str(val).strip().lower() in ("true", "1", "yes", "t")
            elif col == "rating" or col.endswith("_count") or col.endswith("_min"):
                rec[col] = int(float(val)) if not _is_empty(val) else None
            elif col == "confidence":
                rec[col] = float(str(val).replace(",", ".")) if not _is_empty(val) else None
            elif col in ("created_at", "date", "deadline", "closed_at", "last_reopened_at", "birthday"):
                rec[col] = pd.to_datetime(val) if not _is_empty(val) else None
            else:
                rec[col] = None if _is_empty(val) else val
        out_rows.append(rec)
    return pd.DataFrame(out_rows, columns=cols)


def upload_tables() -> None:
    if not report.ok:
        raise RuntimeError("Сначала исправьте ошибки validate (ячейка выше)")

    engine = create_engine(get_settings().sync_database_url)
    with engine.begin() as conn:
        for table in TABLES_TO_LOAD:
            if table not in data:
                print(f"skip {table}: нет в Excel")
                continue
            df = prepare_for_db(data[table], table)
            if TRUNCATE_BEFORE:
                conn.execute(text(f'TRUNCATE TABLE "{table}" RESTART IDENTITY CASCADE'))
                print(f"truncate {table}")
            df.to_sql(table, conn, if_exists="append", index=False, method="multi", chunksize=200)
            n = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
            print(f"OK {table}: залито {len(df)} строк, в БД сейчас {n}")

    print("Готово.")


upload_tables()

## Шаблон листов Excel

| Лист | Обязательные столбцы |
|------|----------------------|
| clients | id, username, email, password_hash |
| employees | id, username, full_name, password_hash |
| tickets | id, client_id, product, date, deadline |
| attachments | id, ticket_id, file_url |
| reviews | id, client_id, rating |
| notifications | id, client_id, ticket_id, type |
| chat_history | id, chat_id, client_id, role, message |

`ticket_id` в reviews и chat_history может быть пустым. `chat_id` — любой UUID, один на диалог.